<a href="https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Lane 2 — Refresh / Content Opportunity Scoring. I picked this lane because it maps most directly onto the decision FlyRank actually cares about: out of thousands of published pages, which one should a reviewer open first when they only have capacity for a handful. It's a ranking problem with a concrete, transparent rule already in place to beat, which means "better" can be measured rather than argued — precision@K against the baseline score, not accuracy against a vague notion of quality. The starter pipeline also gives me a working end-to-end example to study and improve on rather than a blank page, so my effort goes into defining an honest forward-looking target and validating it properly instead of into plumbing. And the output is something a human can act on: a ranked queue where every page carries reason codes explaining why it surfaced.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

The decision this improves: Which content pages a reviewer opens first, out of 30,000, when they have capacity for only a few dozen per cycle.

Who acts, and what they do: A content reviewer or SEO editor working a capacity-limited queue. They open a flagged page, read its reason code, and decide whether to refresh, expand, consolidate, or leave it alone. The model never edits anything — it only orders the queue.

Cost of a wrong call:

False positive — a page ranks high but wasn't worth opening. Cost: wasted reviewer time.
False negative — a genuinely declining page never reaches the top-K. Cost: it keeps losing visibility until someone catches it by chance or a later scoring pass.

These costs are not symmetric. Reviewer capacity is the binding constraint, so false positives are the more expensive error, and I optimise for precision@K — of the K pages surfaced, how many were genuinely worth the time. I assume one review cycle handles about 50 pages, so K = 50.

Why a plain rule isn't enough: The hand-written flag for "declining with demand" fires on 13,152 of 30,000 pages — 43.8% of the inventory. Against a capacity of ~50, a flag that fires 13,152 times gives a reviewer no ordering at all. The problem isn't detecting decline; it's ranking it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [5]:
import os, subprocess
import pandas as pd

REPO_URL = "https://github.com/AaronL123/Flyrank-ML-assignments"
REPO_DIR = "/content/Flyrank-ML-assignments"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

n = len(df)
print(f"Pages: {n:,}   Clients: {df.client_id.nunique()}")

# 1. A hand-written flag fires on nearly half the inventory
declining = ((df.trend_direction == "down") & (df.impressions_90d >= 100)).sum()
print(f'"declining with demand" flags: {declining:,} pages ({declining/n:.1%})')

# 2. Attention is worth concentrating
top10 = df.impressions_90d.sort_values(ascending=False).head(n // 10).sum()
print(f"Top 10% of pages hold {top10/df.impressions_90d.sum():.1%} of impressions")

# 3. Gotcha check: avg_position == 0 means "no data", not rank one
print(f"Rows with avg_position = 0 (no data): {(df.avg_position == 0).sum():,}")

Pages: 30,000   Clients: 32
"declining with demand" flags: 13,152 pages (43.8%)
Top 10% of pages hold 70.2% of impressions
Rows with avg_position = 0 (no data): 1,205


## 4. Careful words: what I can and can't claim

What I can claim:

Observed: on the 30,000-row anonymized starter slice, a hand-written "declining with demand" flag fires on 43.8% of pages, and impressions concentrate heavily — the top 10% of pages carry 70.2% of them.
Measured: any improvement my ranking makes over the rule baseline will be reported as precision@50 under client-holdout validation, so the model is scored on clients it never trained on.
Decision-support: the output is a ranked queue for a human reviewer, with a reason code on every page. It orders attention; it does not act.

What I can't claim:

That refreshing a flagged page causes traffic to recover. That needs an experiment or a causal design, and this data supports neither.
Anything about Google's ranking algorithm. These are FlyRank's own observed search and engagement measurements, not evidence about how ranking works.
That the starter label is the right target. is_declining_label comes from trend_direction, which comes from trend_pct — a current-window bucket, not a future outcome. It's a working proxy, and a stronger version moves to a forward-looking label: features from the prior 90 days, decline measured over the next 30.

Data rules I'm carrying forward: trend_direction and trend_pct produce the label, so neither can ever be a feature. avg_position = 0 means no data, not rank one. Rate columns (ctr, engagement_rate, scroll_rate, ai_traffic_pct) are ×100 percentages — ctr = 0.76 is 0.76%.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.